<a href="https://colab.research.google.com/github/devangc006/Ai-ML/blob/main/Logistic_Regression_Real_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q kagglehub flask scikit-learn pandas

import glob, os, socket, threading
import numpy as np
import pandas as pd
from flask import Flask, Response, jsonify, request
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

FEATURES = ["variance", "skewness", "curtosis", "entropy"]
COLS = FEATURES + ["class"]          # class: 0 = genuine, 1 = forged

# ---------------- 1. LOAD DATA (Kaggle -> UCI -> OpenML fallbacks) ----------------
def load_data():
    try:
        import kagglehub
        path = kagglehub.dataset_download("shantanudhakadd/bank-note-authentication-uci-data")
        df = pd.read_csv(glob.glob(os.path.join(path, "*.csv"))[0])
        df.columns = COLS
        print("Loaded from Kaggle")
    except Exception as e1:
        print("Kaggle failed (%s). Trying UCI..." % e1)
        try:
            url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
                   "00267/data_banknote_authentication.txt")
            df = pd.read_csv(url, header=None, names=COLS)
            print("Loaded from UCI")
        except Exception as e2:
            print("UCI failed (%s). Trying OpenML..." % e2)
            from sklearn.datasets import fetch_openml
            d = fetch_openml("banknote-authentication", version=1, as_frame=True)
            df = d.frame.copy()
            df.columns = COLS
            df["class"] = (df["class"].astype(int) == df["class"].astype(int).max()).astype(int)
            print("Loaded from OpenML")
    df = df.astype(float)
    return df[FEATURES], df["class"].astype(int)

# ---------------- 2. TRAIN + EVALUATE ----------------
X, y = load_data()
print("Rows:", len(X), "| Forged share: %.2f" % y.mean())
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

pipe = Pipeline([("scaler", StandardScaler()),
                 ("clf", LogisticRegression(solver="liblinear", max_iter=5000))])
grid = GridSearchCV(
    pipe,
    {"clf__C": [0.01, 0.1, 0.5, 1, 5, 10, 50, 100, 500],
     "clf__penalty": ["l1", "l2"],
     "clf__class_weight": [None, "balanced"]},
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)
model = grid.best_estimator_
print("Best params:", grid.best_params_)
print("CV accuracy: %.4f" % grid.best_score_)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
metrics = {"accuracy": accuracy_score(y_test, pred), "precision": precision_score(y_test, pred),
           "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred),
           "roc_auc": roc_auc_score(y_test, proba)}
print("Test metrics:", {k: round(float(v), 4) for k, v in metrics.items()})
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["Genuine", "Forged"]))

def first_correct(label):
    idx = X_test.index[(y_test == label) & (pred == label)][0]
    return {k: float(v) for k, v in X_test.loc[idx].items()}

META = {
    "features": FEATURES,
    "ranges": {f: [float(X[f].min()), float(X[f].max()), float(X[f].median())] for f in FEATURES},
    "samples": {"forged": first_correct(1), "genuine": first_correct(0)},
    "metrics": {k: round(float(v), 4) for k, v in metrics.items()},
}

# ---------------- 3. HTML UI ----------------
HTML = """<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Banknote Authenticator</title>
<style>
  body { font-family: system-ui, sans-serif; margin: 0; background: #f4f6f8; color: #222; }
  header { background: #1f3a5f; color: #fff; padding: 16px 24px; }
  main { display: flex; flex-wrap: wrap; gap: 20px; padding: 20px; }
  #form { flex: 2 1 420px; background: #fff; padding: 16px; border-radius: 10px; }
  #result { flex: 1 1 260px; background: #fff; padding: 16px; border-radius: 10px;
            align-self: flex-start; }
  .row { margin-bottom: 16px; }
  .row label { font-weight: 600; display: block; margin-bottom: 4px; text-transform: capitalize; }
  .row .ctl { display: flex; gap: 10px; align-items: center; }
  .row input[type=range] { flex: 1; }
  .row input[type=number] { width: 110px; padding: 6px; border: 1px solid #bbb; border-radius: 6px; }
  button { padding: 8px 14px; border: 0; border-radius: 6px; background: #1f3a5f;
           color: #fff; cursor: pointer; margin-right: 6px; }
  button.alt { background: #6b7c93; }
  #label { font-size: 30px; font-weight: 700; margin: 10px 0; }
  .bar { height: 18px; background: #e3e8ee; border-radius: 9px; overflow: hidden; }
  #fill { height: 100%; width: 0; transition: width .25s, background .25s; }
  .genuine { color: #2e9e5b; } .forged { color: #c0392b; }
  small { color: #666; }
</style></head>
<body>
<header><h1 style="margin:0;font-size:20px">Banknote Authenticator (Logistic Regression)</h1></header>
<main>
  <section id="form">
    <div style="margin-bottom:16px">
      <button onclick="loadSample('genuine')">Load genuine sample</button>
      <button onclick="loadSample('forged')">Load forged sample</button>
      <button class="alt" onclick="resetAll()">Reset</button>
    </div>
    <div id="fields"></div>
  </section>
  <aside id="result">
    <h3 style="margin-top:0">Prediction</h3>
    <div id="label">...</div>
    <div class="bar"><div id="fill"></div></div>
    <p id="probs"></p><hr>
    <small id="metrics"></small>
    <p><small>Educational demo built on the UCI Banknote Authentication dataset.</small></p>
  </aside>
</main>
<script>
let META = null; const nums = {}, rngs = {}; let timer = null;
async function init() {
  META = await (await fetch("meta")).json();
  const root = document.getElementById("fields");
  META.features.forEach(f => {
    const [mn, mx, med] = META.ranges[f];
    const pad = (mx - mn) * 0.25;
    const row = document.createElement("div"); row.className = "row";
    row.innerHTML = "<label>" + f + " <small>(data range " + mn.toFixed(2) + " to " +
                    mx.toFixed(2) + ")</small></label><div class='ctl'></div>";
    const r = document.createElement("input");
    r.type = "range"; r.min = mn - pad; r.max = mx + pad; r.step = "any"; r.value = med;
    const n = document.createElement("input");
    n.type = "number"; n.step = "any"; n.value = med;
    r.addEventListener("input", () => { n.value = (+r.value).toFixed(4); schedule(); });
    n.addEventListener("input", () => { r.value = n.value; schedule(); });
    row.querySelector(".ctl").append(r, n);
    root.appendChild(row);
    rngs[f] = r; nums[f] = n;
  });
  const m = META.metrics;
  document.getElementById("metrics").textContent =
    "Test set: accuracy " + m.accuracy + ", precision " + m.precision +
    ", recall " + m.recall + ", AUC " + m.roc_auc;
  predict();
}
function setVals(getter) {
  META.features.forEach(f => { const v = getter(f); nums[f].value = (+v).toFixed(4); rngs[f].value = v; });
  predict();
}
function loadSample(kind) { setVals(f => META.samples[kind][f]); }
function resetAll() { setVals(f => META.ranges[f][2]); }
function schedule() { clearTimeout(timer); timer = setTimeout(predict, 150); }
async function predict() {
  const payload = {};
  for (const f of META.features) {
    if (nums[f].value === "") return;
    payload[f] = nums[f].value;
  }
  const res = await fetch("predict", { method: "POST",
    headers: { "Content-Type": "application/json" }, body: JSON.stringify(payload) });
  const out = await res.json();
  const label = document.getElementById("label");
  if (out.error) { label.textContent = out.error; return; }
  label.textContent = out.label;
  label.className = out.label === "Forged" ? "forged" : "genuine";
  const pct = out.probability_forged * 100;
  const fill = document.getElementById("fill");
  fill.style.width = pct + "%";
  fill.style.background = pct >= 50 ? "#c0392b" : "#2e9e5b";
  document.getElementById("probs").innerHTML =
    "Forged: <b>" + pct.toFixed(2) + "%</b><br>Genuine: <b>" +
    (out.probability_genuine * 100).toFixed(2) + "%</b>";
}
init();
</script></body></html>"""

# ---------------- 4. FLASK SERVER ----------------
app = Flask(__name__)

@app.route("/")
def home():
    return Response(HTML, mimetype="text/html")

@app.route("/meta")
def get_meta():
    return jsonify(META)

@app.route("/predict", methods=["POST"])
def predict_route():
    data = request.get_json(silent=True) or {}
    try:
        row = [float(data[f]) for f in FEATURES]
    except (KeyError, TypeError, ValueError):
        return jsonify(error="All 4 values must be numeric."), 400
    p = float(model.predict_proba(pd.DataFrame([row], columns=FEATURES))[0, 1])
    return jsonify(label="Forged" if p >= 0.5 else "Genuine",
                   probability_forged=round(p, 4),
                   probability_genuine=round(1 - p, 4))

# ---------------- 5. LAUNCH IN COLAB ----------------
def free_port():
    s = socket.socket(); s.bind(("", 0)); port = s.getsockname()[1]; s.close()
    return port

from google.colab import output
PORT = free_port()
threading.Thread(target=lambda: app.run(host="0.0.0.0", port=PORT, use_reloader=False),
                 daemon=True).start()
output.serve_kernel_port_as_iframe(PORT, height=700)

Kaggle failed (403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.). Trying UCI...
Loaded from UCI
Rows: 1372 | Forged share: 0.44
Best params: {'clf__C': 5, 'clf__class_weight': None, 'clf__penalty': 'l1'}
CV accuracy: 0.9909
Test metrics: {'accuracy': 0.9855, 'precision': 0.9683, 'recall': 1.0, 'f1': 0.9839, 'roc_auc': 1.0}
Confusion matrix:
 [[149   4]
 [  0 122]]
              precision    recall  f1-score   support

     Genuine       1.00      0.97      0.99       153
      Forged       0.97      1.00      0.98       122

    accuracy                           0.99       275
   macro avg       0.98      0.99      0.99       275
weighted avg       0.99      0.99      0.99       275



<IPython.core.display.Javascript object>